In [8]:
%%capture
!pip install --force-reinstall numpy scikit-learn
!pip install torch torchvision torchaudio ultralytics boto3==1.35.55 botocore==1.35.55

# Data Load

In [12]:
import os
import boto3
import botocore

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = 'storage'

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more connection variables are empty.  "
                     "Please check your connection to an S3 bucket.")

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)


def upload_directory_to_s3(local_directory, s3_prefix):
    num_files = 0
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)
            num_files += 1
    return num_files
def upload_file_to_s3(local_file_path, s3_prefix):
    """Upload a single file to S3 bucket with specified prefix"""
    try:
        filename = os.path.basename(local_file_path)
        s3_key = os.path.join(s3_prefix, filename)
        bucket.upload_file(local_file_path, s3_key)
        print(f"{local_file_path} -> {s3_key}")
        return True
    except Exception as e:
        print(f"Error uploading file: {e}")
        return False
        
def download_file_from_s3(s3_key, local_file_path):
    """Download a single file from S3 bucket"""
    try:
        bucket.download_file(s3_key, local_file_path)
        print(f"{s3_key} -> {local_file_path}")
        return True
    except Exception as e:
        print(f"Error downloading file: {e}")
        return False


def delete_file_from_s3(s3_key):
    """Delete a single file from S3 bucket"""
    try:
        s3_object = bucket.Object(s3_key)
        s3_object.delete()
        print(f"Deleted: {s3_key}")
        return True
    except Exception as e:
        print(f"Error deleting file: {e}")
        return False
        
def list_objects(prefix):
    filter = bucket.objects.filter(Prefix=prefix)
    for obj in filter.all():
        print(obj.key)

In [15]:
!rm dataset.zip

In [22]:
download_file_from_s3("datasets/dataset.zip", "./dataset.zip")

datasets/dataset.zip -> ./dataset.zip


True

In [23]:
%%capture
# PPE dataset download 
!rm -rf datasets
!mkdir datasets
!unzip -o dataset.zip -d datasets/
# !rm dataset.zip

In [25]:
import os

# Get the current working directory
cwd = os.getcwd()

# Define your dataset root path
dataset_root = os.path.join(cwd, "datasets")

# Write the dataset.yaml file
yaml_content = f"""path: {dataset_root}
train: images/train
val: images/val
test: images/test

names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: none
  6: Person
  7: no_helmet
  8: no_goggle
  9: no_gloves
  10: no_boots
"""

# Create directory if it doesn't exist
os.makedirs(dataset_root, exist_ok=True)

# Write the file
with open(os.path.join(dataset_root, "dataset.yaml"), "w") as f:
    f.write(yaml_content)

# Model Taining

In [9]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolov8n.pt")  # load a pretrained model (recommended for training)


# Load the dataset

In [10]:
# Train the model
results = model.train(data="datasets/construction-ppe/dataset.yaml", epochs=1, batch=4, imgsz=640)

Ultralytics 8.4.26 🚀 Python-3.12.9 torch-2.11.0+cu130 CPU (AMD EPYC 7R13 Processor)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/construction-ppe/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

FileNotFoundError: [Errno 2] No such file or directory: '/opt/app-root/src/rhelai-cv-ppe-demo/pipelines/datasets/construction-ppe/images/train/image890.jpg'